
# Factor in Practice Part 1: How Can We Predict Stock Prices from Historical Data?

##### "The only thing people learn from history is that humans do not learn from history." - George Bernard Shaw





Are daily stock prices related to their past values?

---
Intraday

---
Short- to medium-term
Use yesterday's stock data to predict today's stock data


In [ ]:
import pandas as pd

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

tickers = ['AAPL', 'MSFT']
start = '2016-01-01'
end = '2022-12-31'

all_data = []
for ticker in tickers:
    stock = yf.Ticker(ticker)
    hist = stock.history(start=start, end=end)
    hist = hist.reset_index()
    hist['Symbol'] = ticker
    info = stock.info
    hist['ShortName'] = info.get('shortName', ticker)
    hist['TradingDate'] = hist['Date'].dt.strftime('%Y-%m-%d')
    hist['Ret'] = hist['Close']
    hist['ChangeRatio'] = hist['Close'].pct_change()
    hist['Amount'] = hist['Volume'] * hist['Close']
    hist['MarketCap'] = hist['Close'] * info.get('sharesOutstanding', 1e9)
    hist['ILLIQ'] = hist['ChangeRatio'].abs() / (hist['Amount'] + 1e-10)
    hist['PE'] = info.get('trailingPE', np.nan)
    hist['PB'] = info.get('priceToBook', np.nan)
    hist['PS'] = info.get('priceToSalesTrailing12Months', np.nan)
    hist['Turnover'] = hist['Volume'] / info.get('sharesOutstanding', 1e9)
    all_data.append(hist)

factor = pd.concat(all_data, axis=0)
factor = factor.sort_values(by=['Symbol', 'TradingDate']).reset_index(drop=True)
factor = factor[['Symbol', 'ShortName', 'TradingDate', 'Ret', 'PE', 'PB', 'PS', 'Turnover', 'MarketCap', 'ChangeRatio', 'Amount', 'ILLIQ']]
factor

In [ ]:
# The factor DataFrame is already sorted and cleaned above
factor

In [ ]:
# Use ChangeRatio as the daily return
df = factor[['Symbol', 'TradingDate', 'ChangeRatio']].rename(columns={'Symbol': 'Stkcd', 'TradingDate': 'Trddt', 'ChangeRatio': 'Dretwd'})
df

In [ ]:
liq = factor[['ILLIQ']].copy()
liq

In [ ]:
liq = liq.drop(columns=["ILLIQ"])
# Keep only the ILLIQ column for merging
liq = factor[['ILLIQ']].copy()
liq

In [ ]:
ret = df.copy()
ret_reset = ret.reset_index(drop=True)
liq_reset = liq.reset_index(drop=True)
factor_reset = factor.reset_index(drop=True)

merged = pd.concat([ret_reset, liq_reset, factor_reset], axis=1)
merged = merged.drop(columns=['TradingDate', 'Symbol'])

df_final = merged.rename(columns={
    'Stkcd': 'Stkcd',
    'Trddt': 'Trddt',
    'Dretwd': 'Dretwd',
    'MarketCap': 'CirculatedMarketValue',
    'ILLIQ': 'ILLIQ'
})

df_final

In [ ]:
X = df_final[['ILLIQ', 'PE', 'PB', 'PS', 'CirculatedMarketValue', 'ChangeRatio', 'Turnover']]
y = df_final['Dretwd']


In [ ]:
X

In [ ]:
y

In [ ]:
# Shift by one day!!!!!!
y_shift = y.shift(-1)
y_shift

In [ ]:
y_shift.fillna(method='ffill', inplace=True)
y_shift

# Use yesterday's stock data to predict today's return direction
# We already have a complete dataset


In [ ]:
X.to_csv('X.csv', index=False)
y_shift.to_csv('y.csv', index=False)